# Stack Chips

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pathlib import Path
from tqdm import tqdm

# ─── Paths ───────────────────────────────────────────────────────────────────
april_dir  = Path("...")
june_dir   = Path("...")
august_dir = Path("...")

out_rg_nir     = Path("...")
#out_rg_nir_swir = Path("...")

out_rg_nir.mkdir(parents=True, exist_ok=True)
#out_rg_nir_swir.mkdir(parents=True, exist_ok=True)

In [ ]:
# ─── Band indices (0-based) ───────────────────────────────────────────────────
# B03=1, B04=0, B08=3, B11=8, B12=9
BANDS_RG_NIR      = [1, 0, 3]           # 3 bands × 3 months = 9 channels
#BANDS_RG_NIR_SWIR = [1, 0, 3, 8, 9]    # 5 bands × 3 months = 15 channels

# ─── Stack ───────────────────────────────────────────────────────────────────
chip_names = sorted([f.name for f in april_dir.glob("*.tif")])
print(f"Found {len(chip_names)} chips")

for chip_name in tqdm(chip_names):
    april_path  = april_dir  / chip_name
    june_path   = june_dir   / chip_name
    august_path = august_dir / chip_name

    # Skip if any month is missing
    if not all([april_path.exists(), june_path.exists(), august_path.exists()]):
        print(f"Skipping {chip_name} — missing in at least one month")
        continue

    arrays = {}
    meta = None
    for month, path in [("april", april_path), ("june", june_path), ("august", august_path)]:
        with rasterio.open(path) as src:
            arrays[month] = src.read()  # shape: (11, H, W) — band 10 is a constant mask, unused 
            if meta is None:
                meta = src.meta.copy()

    for bands, out_dir in [(BANDS_RG_NIR, out_rg_nir)]: #(BANDS_RG_NIR_SWIR, out_rg_nir_swir), 
        stack = np.concatenate([
            arrays["april"][bands],
            arrays["june"][bands],
            arrays["august"][bands],
        ], axis=0)  # shape: (9 or 15, H, W), channel first logic

        out_meta = meta.copy()
        out_meta.update({"count": stack.shape[0], "dtype": stack.dtype})

        with rasterio.open(out_dir / chip_name, "w", **out_meta) as dst:
            dst.write(stack)

print("Done!")

In [ ]:
import os
import shutil
import tifffile
import numpy as np
from glob import glob
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
# Removing unusable SR chips
# ──────────────────────────────────────────────────────────────────────────────
# SRGAN super-resolution provides only incomplete coverage for some chips;
# missing areas are encoded as 0 (no NoData flag).
# Chips for which the proportion of zeros across all bands is >= ZERO_FRAC_THRESHOLD
# are considered unusable and are removed from both band configurations
# (moved to a separate folder, not deleted).
#
# A threshold value of 0.5 was selected following visual inspection: chips with a low
# proportion of zeros (tile edges) are retained; only chips that are largely incomplete
# are excluded.
# ──────────────────────────────────────────────────────────────────────────────

DEST_DIR            = Path('...')
ZERO_FRAC_THRESHOLD = 0.5

out_rg_nir      = Path('...')       # anpassen
out_rg_nir_swir = Path('...')

DEST_DIR.mkdir(parents=True, exist_ok=True)


def zero_fraction(path):
    """Proportion of zero entries across all channels of a chip."""
    img = tifffile.imread(path).astype(np.float32)
    return float((img == 0).mean())


# Determine the criterion solely on the basis of the SWIR chips (the most comprehensive test).
swir_paths = sorted(glob(str(out_rg_nir_swir / '*.tif')))
print(f'Check {len(swir_paths)} Chips in RG_NIR_SWIR ...')

broken = []
for i, p in enumerate(swir_paths):
    if zero_fraction(p) >= ZERO_FRAC_THRESHOLD:
        broken.append(os.path.basename(p))
    if (i + 1) % 2000 == 0:
        print(f'  {i+1}/{len(swir_paths)} ...')

print(f'\nDefective chips (zero percentage >= {ZERO_FRAC_THRESHOLD:.0%}): {len(broken)}')

# Move the same ‘broken’ list from BOTH folders.
moved_rg_nir      = 0
moved_rg_nir_swir = 0

for name in broken:
    src_swir = out_rg_nir_swir / name
    if src_swir.exists():
        (DEST_DIR / 'RG_NIR_SWIR').mkdir(exist_ok=True)
        shutil.move(str(src_swir), str(DEST_DIR / 'RG_NIR_SWIR' / name))
        moved_rg_nir_swir += 1

    src_rgnir = out_rg_nir / name
    if src_rgnir.exists():
        (DEST_DIR / 'RG_NIR').mkdir(exist_ok=True)
        shutil.move(str(src_rgnir), str(DEST_DIR / 'RG_NIR' / name))
        moved_rg_nir += 1

print(f'\nMoved from RG_NIR_SWIR: {moved_rg_nir_swir}')
print(f'Moved from RG_NIR:      {moved_rg_nir}')
print(f'Destination folder: {DEST_DIR}')

In [ ]:
from pathlib import Path
import shutil

LABEL_DIR    = Path('...')
DEST_DIR_lbl = Path('...')

DEST_DIR_lbl.mkdir(parents=True, exist_ok=True)

moved_lbls = 0
for name in broken:                      # broken from the SR cleanup script
    src = LABEL_DIR / name
    if src.exists():
        shutil.move(str(src), str(DEST_DIR_lbl / name))
        moved_lbls += 1

print(f'\nMoved from Labels_filtered: {moved_lbls}')
print(f'Destination folder: {DEST_DIR_lbl}')

In [ ]:
"""Moves label TIFs without an associated SR chip to a separate folder."""

from pathlib import Path
import shutil

LABELS_DIR = Path("...")
SRCHIP_DIR = Path("...")
ORPHAN_DIR = Path("...")

DRY_RUN = False  # Check first, then set to False

ORPHAN_DIR.mkdir(parents=True, exist_ok=True)

# Chip base names (without suffix) as a set
chip_stems = {p.stem for p in SRCHIP_DIR.glob("*.tif")}

moved = 0
for label in LABELS_DIR.glob("*.tif"):
    if label.stem not in chip_stems:
        print(("[DRY] " if DRY_RUN else "") + f"move {label.name}")
        if not DRY_RUN:
            shutil.move(str(label), str(ORPHAN_DIR / label.name))
        moved += 1

print(f"\n{moved} Labels without an SR chip "
      f"({'would be postponed' if DRY_RUN else 'postponed'}).")